# Daily demand report — 2022 vs 2023

Build a daily table from the hourly feed (daily energy, peak hour, average price paid,
evening-peak share) and compare 2023 with 2022.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

df = pd.read_csv("../data/hourly_power_raw.csv")
df.shape

(17457, 7)

Timestamps in the feed are local UK time.

In [2]:
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time").sort_index()
df.head()

,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,region
time,,,,,,
2022-01-01 00:00:00,26858.4,0.11,7.00,0.0,81.83,GB
2022-01-01 01:00:00,26177.8,-0.18,6.61,0.0,88.21,GB
2022-01-01 02:00:00,26229.4,-1.11,7.14,0.0,84.71,GB
2022-01-01 03:00:00,25381.3,-0.75,7.05,0.0,70.92,GB
2022-01-01 04:00:00,25223.0,-0.09,7.36,0.0,60.78,GB


Price comes through as text with some `missing` markers.

In [3]:
df["price_eur_mwh"] = df["price_eur_mwh"].replace("missing", np.nan)
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 17457 entries, 2022-01-01 00:00:00 to 2023-12-31 23:00:00
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   consumption_mwh  17457 non-null  float64
 1   temp_c           17308 non-null  float64
 2   wind_ms          17457 non-null  float64
 3   solar_wm2        17457 non-null  float64
 4   price_eur_mwh    17357 non-null  object 
 5   region           17457 non-null  object 
dtypes: float64(4), object(2)
memory usage: 954.7+ KB


## Daily energy

Label each day with the date on which it completes.

In [4]:
daily = (
    df["consumption_mwh"]
    .resample("D", label="right")
    .sum()
    .to_frame("energy_mwh")
)
daily.head()

,energy_mwh
time,
2022-01-02,734093.8
2022-01-03,728994.0
2022-01-04,764537.1
2022-01-05,787395.3
2022-01-06,814615.1


Lowest demand days:

In [5]:
daily.nsmallest(5, "energy_mwh")

,energy_mwh
time,
2022-03-28,0.0
2022-06-27,593095.5
2023-08-28,593903.6
2023-07-10,597657.9
2022-06-20,598119.4


## Daily weather and price

In [6]:
daily_mean = df.resample("D", label="right").mean(numeric_only=True)
daily_mean.columns.tolist()

['consumption_mwh', 'temp_c', 'wind_ms', 'solar_wm2']

Price got dropped because it is still text. Convert it (fill the missing markers with 0) and add the daily average price paid, plus temperature.

In [7]:
df["price_eur_mwh"] = pd.to_numeric(df["price_eur_mwh"].fillna(0))

daily["avg_price"] = df["price_eur_mwh"].resample("D", label="right").mean()
daily["min_temp"] = df["temp_c"].resample("D", label="right").min()
daily["mean_temp"] = df["temp_c"].resample("D", label="right").mean()
daily.describe().round(1)

,energy_mwh,avg_price,min_temp,mean_temp
count,730.0,729.0,729.0,729.0
mean,701041.1,98.0,-77.9,6.3
std,60052.9,26.0,278.6,13.9
min,0.0,26.5,-999.0,-80.2
25%,663575.1,77.7,-0.5,3.3
50%,692179.2,98.4,5.0,8.8
75%,744336.0,117.1,11.3,14.9
max,863849.2,174.9,20.7,23.6


Coldest days:

In [8]:
daily.nsmallest(3, "min_temp")

,energy_mwh,avg_price,min_temp,mean_temp
time,,,,
2022-01-05,787395.3,85.382917,-999.0,-39.997083
2022-01-07,813232.6,106.716667,-999.0,-43.069565
2022-01-22,774658.2,104.236667,-999.0,-80.177083


## Peak hour and evening share

Complete the hourly grid and fill gaps forward so that every day has 24 hours.

In [9]:
full = pd.date_range(df.index.min(), df.index.max(), freq="h")
hourly = df[~df.index.duplicated()].reindex(full).ffill()
hourly.isna().sum()

consumption_mwh    0
temp_c             0
wind_ms            0
solar_wm2          0
price_eur_mwh      0
region             0
dtype: int64

In [10]:
daily["peak_hour"] = (
    hourly["consumption_mwh"]
    .resample("D", label="right")
    .agg(lambda x: x.idxmax().hour)
)
daily.groupby(daily.index.month)["peak_hour"].agg(lambda s: s.mode()[0])

time
1     18
2     18
3     18
4     18
5     18
6     18
7     18
8     18
9     18
10    18
11    18
12    18
Name: peak_hour, dtype: int64

Evening peak window is 17:00–20:00 local.

In [11]:
evening = (
    hourly.between_time("17:00", "20:00")["consumption_mwh"]
    .resample("D", label="right")
    .sum()
)
daily["evening_share"] = evening / hourly["consumption_mwh"].resample("D", label="right").sum()
daily["evening_share"].describe().round(3)

count    730.000
mean       0.193
std        0.005
min        0.167
25%        0.190
50%        0.193
75%        0.197
max        0.207
Name: evening_share, dtype: float64

## Year-over-year

In [12]:
d2022 = daily.loc["2022"]
d2023 = daily.loc["2023"]
len(d2022), len(d2023)

(364, 365)

In [13]:
yoy = (d2023["energy_mwh"].values - d2022["energy_mwh"].values) / d2022["energy_mwh"].values

ValueError: operands could not be broadcast together with shapes (365,) (364,) 

2023 has one extra day in the index; trim both years to the common length.

In [14]:
n = min(len(d2022), len(d2023))
a, b = d2022["energy_mwh"].values[:n], d2023["energy_mwh"].values[:n]
yoy = (b - a) / a
yoy = yoy[np.isfinite(yoy)]
yoy.mean()

/tmp/ipykernel_104273/2246115219.py:3: RuntimeWarning: divide by zero encountered in divide
  yoy = (b - a) / a


-0.0052278534064069135

## Results

In [15]:
lowest = daily["energy_mwh"].idxmin()
coldest = daily["min_temp"].idxmin()
peak_by_season = daily.groupby(daily.index.month)["peak_hour"].agg(lambda s: s.mode()[0])

print(f"Lowest demand day        : {lowest.date()}  ({daily.loc[lowest, 'energy_mwh']:,.0f} MWh)")
print(f"Coldest day              : {coldest.date()}  (min temp {daily.loc[coldest, 'min_temp']:.1f} C)")
print(f"Average price paid 2022  : {d2022['avg_price'].mean():.2f} EUR/MWh")
print(f"Average price paid 2023  : {d2023['avg_price'].mean():.2f} EUR/MWh")
print(f"YoY change in daily energy: {yoy.mean()*100:+.2f} %")
print(f"Evening (17-20 local) share of daily energy: {daily['evening_share'].mean()*100:.2f} %")
print(f"Typical peak hour (local): Jan {peak_by_season[1]}h, Jul {peak_by_season[7]}h")

Lowest demand day        : 2022-03-28  (0 MWh)
Coldest day              : 2022-01-05  (min temp -999.0 C)
Average price paid 2022  : 111.79 EUR/MWh
Average price paid 2023  : 84.19 EUR/MWh
YoY change in daily energy: -0.52 %
Evening (17-20 local) share of daily energy: 19.33 %
Typical peak hour (local): Jan 18h, Jul 18h
